# 05 — Camera Movement Estimation (Lucas-Kanade Optical Flow)

When the broadcast camera pans or tilts, every pixel in the frame moves — including player bounding boxes.  
Without compensation, a player standing still would appear to "move" as the camera pans past them.

**Approach:**
1. Pick feature points along the static pitch sidelines (left and right edges of the frame)
2. Track those points between consecutive frames using Lucas-Kanade optical flow
3. The motion of *static* sideline features = camera motion
4. Subtract that motion from every player's position

**Connection to the pipeline:** `football_analysis/camera_movement_estimator/camera_movement_estimator.py`

In [ ]:
import sys
sys.path.append('..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [ ]:
VIDEO_PATH = '../input_videos/08fd33_4.mp4'

cap = cv2.VideoCapture(VIDEO_PATH)
frames_bgr = []
for _ in range(60):   # first 60 frames is enough to see camera movement
    ret, f = cap.read()
    if ret:
        frames_bgr.append(f)
cap.release()
print(f'Loaded {len(frames_bgr)} frames')

## Why mask to the sidelines?

We only want to track points that are *guaranteed* to be on static pitch markings — not on players (who move independently).  
The pipeline masks feature detection to thin strips on the left (cols 0-20) and right (cols 900-1050) edges of the frame, where the sideline boundaries are.

In [ ]:
h, w = frames_bgr[0].shape[:2]

mask = np.zeros((h, w), dtype=np.uint8)
mask[:, 0:20]    = 255   # left sideline strip
mask[:, 900:1050] = 255  # right sideline strip

fig, ax = plt.subplots(figsize=(14, 6))
vis = frames_bgr[0].copy()
vis[mask == 0] = (vis[mask == 0] * 0.3).astype(np.uint8)   # dim non-mask areas
ax.imshow(vis[:, :, ::-1])
ax.set_title('Feature detection mask (bright strips = where we look for points)')
ax.axis('off')
plt.tight_layout()
plt.show()

## Finding good features to track

`cv2.goodFeaturesToTrack` (Shi-Tomasi) finds corner-like regions — points with gradient in both x and y directions.  
These are stable and easy to re-locate in the next frame.

In [ ]:
lk_params = dict(
    winSize=(15, 15),
    maxLevel=2,
    criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03),
)
feature_params = dict(maxCorners=100, qualityLevel=0.3, minDistance=3, blockSize=7)

gray0 = cv2.cvtColor(frames_bgr[0], cv2.COLOR_BGR2GRAY)
p0    = cv2.goodFeaturesToTrack(gray0, mask=mask, **feature_params)

print(f'Found {len(p0)} feature points')

vis = frames_bgr[0].copy()
for pt in p0:
    x, y = pt.ravel().astype(int)
    cv2.circle(vis, (x, y), 4, (0, 255, 0), -1)

plt.figure(figsize=(14, 6))
plt.imshow(vis[:, :, ::-1])
plt.title(f'Feature points ({len(p0)} found in sideline strips)')
plt.axis('off')
plt.show()

## Lucas-Kanade optical flow

`cv2.calcOpticalFlowPyrLK` tracks each feature point from frame N to frame N+1.  
It uses a pyramid of image resolutions (`maxLevel=2`) to handle large displacements.  
The output `status` flag tells us which points were successfully tracked.

In [ ]:
gray1 = cv2.cvtColor(frames_bgr[1], cv2.COLOR_BGR2GRAY)
p1, status, _ = cv2.calcOpticalFlowPyrLK(gray0, gray1, p0, None, **lk_params)

good0 = p0[status == 1]
good1 = p1[status == 1]

print(f'{len(good0)} of {len(p0)} points tracked successfully')

# Visualise flow vectors
vis = frames_bgr[1].copy()
for a, b in zip(good0, good1):
    x0, y0 = a.ravel().astype(int)
    x1, y1 = b.ravel().astype(int)
    cv2.arrowedLine(vis, (x0, y0), (x1, y1), (0, 255, 0), 2, tipLength=0.5)

plt.figure(figsize=(14, 6))
plt.imshow(vis[:, :, ::-1])
plt.title('Optical flow vectors (frame 0 → frame 1)')
plt.axis('off')
plt.show()

## Extracting camera displacement (dx, dy)

The *mean* displacement of all successfully tracked points = camera motion.  
We use the mean rather than any single point to be robust to tracking errors.  

We also apply a `minimum_distance=5` threshold — tiny movements (sub-pixel, wind shake) are treated as zero to avoid noise accumulation.

In [ ]:
MINIMUM_DISTANCE = 5
camera_movement = []  # (dx, dy) per frame transition

prev_gray  = cv2.cvtColor(frames_bgr[0], cv2.COLOR_BGR2GRAY)
prev_pts   = cv2.goodFeaturesToTrack(prev_gray, mask=mask, **feature_params)

for i in range(1, len(frames_bgr)):
    curr_gray = cv2.cvtColor(frames_bgr[i], cv2.COLOR_BGR2GRAY)
    curr_pts, st, _ = cv2.calcOpticalFlowPyrLK(prev_gray, curr_gray, prev_pts, None, **lk_params)

    if curr_pts is not None and st is not None:
        good_prev = prev_pts[st == 1]
        good_curr = curr_pts[st == 1]
        diffs = good_curr - good_prev
        dx, dy = diffs.mean(axis=0) if len(diffs) else (0, 0)
    else:
        dx, dy = 0, 0

    if abs(dx) < MINIMUM_DISTANCE:
        dx = 0
    if abs(dy) < MINIMUM_DISTANCE:
        dy = 0

    camera_movement.append((dx, dy))

    prev_gray = curr_gray
    prev_pts  = cv2.goodFeaturesToTrack(curr_gray, mask=mask, **feature_params)
    if prev_pts is None:
        prev_pts = good_curr.reshape(-1, 1, 2) if len(good_curr) else prev_pts

print(f'Computed {len(camera_movement)} displacement vectors')

In [ ]:
dxs = [d[0] for d in camera_movement]
dys = [d[1] for d in camera_movement]

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
axes[0].plot(dxs, color='steelblue')
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].set_ylabel('dx (pixels)')
axes[0].set_title('Camera movement — horizontal')

axes[1].plot(dys, color='tomato')
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_ylabel('dy (pixels)')
axes[1].set_xlabel('Frame')
axes[1].set_title('Camera movement — vertical')

plt.tight_layout()
plt.show()

## How this corrects player positions

In `cam_est.add_adjust_positions_to_tracks()`, we subtract the *cumulative* camera displacement from each player's raw pixel position.  
This gives us positions **as if the camera never moved** — essential for accurate speed/distance later.

In [ ]:
# Cumulative displacement
cum_dx = np.cumsum([0] + dxs)
cum_dy = np.cumsum([0] + dys)

plt.figure(figsize=(14, 4))
plt.plot(cum_dx, label='Cumulative dx', color='steelblue')
plt.plot(cum_dy, label='Cumulative dy', color='tomato')
plt.axhline(0, color='black', linewidth=0.5)
plt.xlabel('Frame')
plt.ylabel('Pixels')
plt.title('Cumulative camera movement')
plt.legend()
plt.tight_layout()
plt.show()

## Key takeaways

| Concept | Detail |
|---|---|
| Optical flow | Tracks small image patches between frames using brightness constancy assumption |
| Sideline mask | Guarantees we only track static background, not moving players |
| Minimum distance | Filters sub-pixel noise to avoid drift accumulation |
| Subtraction | Camera displacement subtracted from each player's foot position |

**Next:** `06_perspective_transform.ipynb` — converting pixel positions to real-world metres